# Functions only

In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

# ============================================================================
# GLOBALS / ENVIRONMENT
# ============================================================================

ALLOWED_DATASETS = ("STU00225089",)
DATASET_NAME_CATATONIA = "STU00225089"
_APARC_MEASURES = {"volume", "thickness", "meancurv"}


def _require_env_path(name: str) -> Path:
    value = os.getenv(name)

    if not value:
        raise EnvironmentError(f"Required environment variable '{name}' is not set.")

    return Path(value)


def add_base_subjid(df: pd.DataFrame, source_col: str = "subjid") -> pd.DataFrame:
    
    df = df.copy()

    if source_col not in df.columns:
        raise KeyError(f"Required subject-ID column '{source_col}' is missing.")

    df[source_col] = df[source_col].astype("string").str.strip()

    if df[source_col].isna().any():
        bad_rows = df.index[df[source_col].isna()].tolist()
        raise ValueError(f"Missing subject IDs in column '{source_col}' at rows: {bad_rows}")

    if df[source_col].eq("").any():
        bad_rows = df.index[df[source_col].eq("")].tolist()
        raise ValueError(f"Empty subject IDs in column '{source_col}' at rows: {bad_rows}")

    df["subjid_base"] = df[source_col].str.replace(r"-\d{3}$", "", regex=True)

    return df


def load_aparcaseg_table(fn: str) -> pd.DataFrame:
    """Load an aparc+aseg E-field stats table while preserving modeled IDs."""

    path = Path(fn)

    if not path.is_file():
        raise FileNotFoundError(f"aparc+aseg stats table does not exist: {path}")

    df = pd.read_csv(path, sep="\t")
    
    if "subject_id" in df.columns and "subjid" not in df.columns:
        df = df.rename(columns={"subject_id": "subjid"})

    if "subjid" not in df.columns:
        raise KeyError("Expected either 'subjid' or 'subject_id' column.")

    df = add_base_subjid(df, source_col="subjid")

    duplicate_mask = df["subjid"].duplicated(keep=False)

    if duplicate_mask.any():
        duplicates = df.loc[duplicate_mask, "subjid"].tolist()
        raise ValueError(f"Duplicate modeled subject IDs found in {path}: {duplicates}")

    roi_cols = [col for col in df.columns if col not in {"subjid", "subjid_base"}]
    df[roi_cols] = df[roi_cols].apply(pd.to_numeric, errors="raise")

    return df


def get_stats_table_files(
    results_dir: str,
    field_name: str = "magnE_mean",
    measure: str = "mean",
    static_step: int = 0,
    adaptive_step: int = 2) -> dict:

    results_dir = Path(results_dir)

    if not results_dir.is_dir():
        raise NotADirectoryError(f"Results directory does not exist: {results_dir}")

    filename = f"aparc+aseg.{field_name}.{measure}.stats.table"
    files = {}

    for model in ("skin_single", "skin_double"):
        model_dir = results_dir / model / "stats"
        static_file = model_dir / f"step_{static_step}" / filename
        adaptive_file = model_dir / f"step_{adaptive_step}" / filename

        missing_files = [path for path in (static_file, adaptive_file) if not path.is_file()]

        if missing_files:
            raise FileNotFoundError("Missing expected stats table(s):\n" + "\n".join(str(path) for path in missing_files))

        files[model] = {
            "static": str(static_file),
            "adaptive": str(adaptive_file),
        }

    return files


def load_stimulus(filepath: Path = None) -> pd.DataFrame:
    """
    Currently unused in the main dataframe build.
    Retained for later need-based merging.
    """

    if filepath is None:
        raise ValueError("filepath must be provided.")

    filepath = Path(filepath)

    if not filepath.is_file():
        raise FileNotFoundError(f"Stimulus file does not exist: {filepath}")

    df_stim = pd.read_excel(filepath, sheet_name="stimulus")

    required_cols = ["subjid", "date", "pulse_width_ms", "frequency_hz", "percent_charge"]
    missing_cols = [col for col in required_cols if col not in df_stim.columns]

    if missing_cols:
        raise KeyError(f"Stimulus table is missing required columns: {missing_cols}")

    col = "subjid"
    idx_bad_rows = df_stim.index[df_stim[col].isna()].tolist()

    print(f"Dropping {len(idx_bad_rows)} rows due to missing values in '{col}': {idx_bad_rows}")

    df_stim = df_stim.drop(index=idx_bad_rows).copy()
    df_stim["subjid_org"] = df_stim["subjid"].copy()
    df_stim = add_base_subjid(df_stim, source_col="subjid")

    df_stim["pulse_width_ms"] = df_stim["pulse_width_ms"].replace({
        5: 0.5,
        25: 0.25,
        50: 0.50,
        75: 0.75,
    })

    return df_stim


def load_cogscores_batch(fn: str) -> pd.DataFrame:
    path = Path(fn)

    if not path.is_file():
        raise FileNotFoundError(f"Cognitive-score file does not exist: {path}")

    df = pd.read_excel(path)

    cleaned_columns = df.columns.str.strip()

    if cleaned_columns.duplicated().any():
        duplicates = cleaned_columns[cleaned_columns.duplicated()].tolist()
        raise ValueError(f"Cleaning whitespace creates duplicate columns: {duplicates}")

    df.columns = cleaned_columns

    required_cols = ["subjid", "cgi_start", "cgi_end", "date_start", "date_end (acute)", "mrn"]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise KeyError(f"Cognitive-score table is missing required columns: {missing_cols}")

    for column in ("cgi_start", "cgi_end"):
        if df[column].isna().any():
            idx_bad_rows = df.index[df[column].isna()].tolist()
            print(f"Dropping {len(idx_bad_rows)} rows due to missing values in '{column}': {idx_bad_rows}")
            df = df.drop(index=idx_bad_rows).copy()

    df["subjid_org"] = df["subjid"].copy()
    df = add_base_subjid(df, source_col="subjid")
    df = df.drop(columns=["mrn"])

    return df


def df_combine_stim_cog(
    df_stim: pd.DataFrame,
    df_cog: pd.DataFrame,
    output_path: str | None = "data_model.csv") -> pd.DataFrame:
    """
    Currently unused in the main dataframe build.
    Retained for later need-based merging.
    """

    df_stim = df_stim.copy()
    df_cog = df_cog.copy()

    required_stim_cols = {"subjid", "subjid_base", "date"}
    required_cog_cols = {"subjid", "subjid_base", "date_start", "date_end (acute)"}

    missing_stim = required_stim_cols - set(df_stim.columns)
    missing_cog = required_cog_cols - set(df_cog.columns)

    if missing_stim:
        raise KeyError(f"df_stim missing required columns: {sorted(missing_stim)}")

    if missing_cog:
        raise KeyError(f"df_cog missing required columns: {sorted(missing_cog)}")

    stim_subjects = set(df_stim["subjid_base"].dropna())
    cog_subjects = set(df_cog["subjid_base"].dropna())

    only_stim = sorted(stim_subjects - cog_subjects)
    only_cog = sorted(cog_subjects - stim_subjects)

    if only_stim or only_cog:
        raise ValueError(
            "Subject mismatch between stimulus and cognitive data.\n"
            f"Only in stimulus ({len(only_stim)}): {only_stim}\n"
            f"Only in cognitive ({len(only_cog)}): {only_cog}"
        )

    date_columns = [
        (df_stim, "date"),
        (df_cog, "date_start"),
        (df_cog, "date_end (acute)"),
    ]

    for dataframe, column in date_columns:
        if dataframe[column].isna().any():
            bad_rows = dataframe.index[dataframe[column].isna()].tolist()
            raise ValueError(f"Missing date values in '{column}' at rows: {bad_rows}")

        dataframe[column] = pd.to_datetime(dataframe[column], format="mixed", errors="raise").dt.normalize()

    df_model = df_stim.merge(
        df_cog,
        on="subjid_base",
        how="inner",
        suffixes=("_stim", "_cog"),
        validate="many_to_many",
    )

    if df_model.empty:
        raise ValueError("Stimulus/cognitive merge produced zero rows.")

    in_window = df_model["date"].between(df_model["date_start"], df_model["date_end (acute)"], inclusive="both")

    n_before = len(df_model)
    n_outside = int((~in_window).sum())

    df_model = df_model.loc[in_window].copy()
    df_model = df_model.sort_values(["subjid_base", "date"]).reset_index(drop=True)

    if df_model.empty:
        raise ValueError("No stimulation sessions remain after applying the acute-treatment date windows.")

    if n_outside > 0:
        print(
            f"df_combine_stim_cog: intentionally excluded {n_outside}/{n_before} merged row combinations "
            "because the stimulation date was outside the corresponding acute-treatment window."
        )

    df_model = df_model.replace(r"^\s*$", np.nan, regex=True)
    df_model["cgi"] = np.nan

    start_mask = df_model["date"].eq(df_model["date_start"])
    end_mask = df_model["date"].eq(df_model["date_end (acute)"])

    df_model.loc[start_mask, "cgi"] = df_model.loc[start_mask, "cgi_start"]
    df_model.loc[end_mask, "cgi"] = df_model.loc[end_mask, "cgi_end"]

    if output_path is not None:
        df_model.to_csv(output_path, index=False)

    return df_model

def df_compute_ef_sf(df: pd.DataFrame=None) -> pd.DataFrame:
    """ef_sf = pulse_width_ms * frequency_hz * percent_charge."""

    df = df.copy()

    sf_columns = ["pulse_width_ms", "frequency_hz", "percent_charge"]
    missing_cols = [col for col in sf_columns if col not in df.columns]

    if missing_cols:
        raise KeyError(f"Missing EF scaling-factor columns: {missing_cols}")

    df[sf_columns] = df[sf_columns].apply(pd.to_numeric, errors="raise")

    if df[sf_columns].isna().any().any():
        bad_rows = df.index[df[sf_columns].isna().any(axis=1)].tolist()
        raise ValueError(f"Missing values found in EF scaling-factor columns at rows: {bad_rows}")

    df["ef_sf"] = df["pulse_width_ms"] * df["frequency_hz"] * df["percent_charge"]

    return df


def df_compute_change_cgi(df: pd.DataFrame) -> pd.DataFrame:
    """Compute cgi_change = cgi_end - cgi_start."""

    required_cols = ["subjid", "cgi_start", "cgi_end", "response_type", "session_num"]
    missing_cols = [col for col in required_cols if col not in df.columns]

    if missing_cols:
        raise KeyError(f"Required columns are missing: {missing_cols}")

    result = df.loc[:, required_cols].copy()

    result["cgi_start"] = pd.to_numeric(result["cgi_start"], errors="raise")
    result["cgi_end"] = pd.to_numeric(result["cgi_end"], errors="raise")

    if result[["cgi_start", "cgi_end"]].isna().any().any():
        raise ValueError("Missing CGI start or end values detected.")

    result["cgi_change"] = result["cgi_end"] - result["cgi_start"]

    return result


def add_roi_stats(df_model: pd.DataFrame, df_stats: pd.DataFrame) -> pd.DataFrame:
    """
    Attach clinical/person-level information to every modeled MRI/FEM scan.

    df_stats is the primary dataframe so modeled subjects are never lost
    simply because clinical data are unavailable.

    subjid:
        exact modeled MRI/FEM ID

    subjid_base:
        person-level ID

    subjid_cog:
        original ID from cognitive-score table
    """

    df_model = df_model.copy()
    df_stats = df_stats.copy()

    required_model_cols = {"subjid_base"}
    required_stats_cols = {"subjid", "subjid_base"}

    missing_model = required_model_cols - set(df_model.columns)
    missing_stats = required_stats_cols - set(df_stats.columns)

    if missing_model:
        raise KeyError(f"df_model missing required columns: {sorted(missing_model)}")

    if missing_stats:
        raise KeyError(f"df_stats missing required columns: {sorted(missing_stats)}")

    duplicate_mask = df_stats["subjid"].duplicated(keep=False)

    if duplicate_mask.any():
        duplicates = df_stats.loc[duplicate_mask, "subjid"].tolist()
        raise ValueError(f"Modeled subject IDs must be unique within an E-field stats table. Duplicates: {duplicates}")

    modeled_subjects = set(df_stats["subjid_base"].dropna())
    clinical_subjects = set(df_model["subjid_base"].dropna())

    modeled_without_clinical = sorted(modeled_subjects - clinical_subjects)
    clinical_without_model = sorted(clinical_subjects - modeled_subjects)

    if modeled_without_clinical:
        print(
            f"add_roi_stats: {len(modeled_without_clinical)} modeled clinical subjects have no cognitive-score data: "
            f"{modeled_without_clinical}"
        )

    if clinical_without_model:
        print(
            f"add_roi_stats: {len(clinical_without_model)} cognitive-score subjects have no model in this dataset: "
            f"{clinical_without_model}"
        )

    if "subjid" in df_model.columns:
        df_model = df_model.rename(columns={"subjid": "subjid_cog"})

    clinical_counts = df_model.groupby("subjid_base").size()
    modeled_counts = df_stats.groupby("subjid_base").size()

    expected_rows = 0

    for subject in modeled_subjects:
        n_models = int(modeled_counts[subject])
        n_clinical = int(clinical_counts[subject]) if subject in clinical_counts.index else 1
        expected_rows += n_models * n_clinical

    result = df_stats.merge(
        df_model,
        on="subjid_base",
        how="left",
        validate="many_to_many",
        suffixes=("", "_cog"),
    )

    if len(result) != expected_rows:
        raise RuntimeError(
            f"Unexpected row count after modeled-to-clinical merge. "
            f"Expected: {expected_rows}, observed: {len(result)}"
        )

    return result


def build_model_roi(df_model: pd.DataFrame, 
                    files_agg: dict, 
                    skin_model: str = "skin_single", 
                    model_type: str = "static") -> pd.DataFrame:

    valid_skin_models = {"skin_single", "skin_double"}
    valid_model_types = {"static", "adaptive"}

    if skin_model not in valid_skin_models:
        raise ValueError(f"skin_model must be one of {sorted(valid_skin_models)}.")

    if model_type not in valid_model_types:
        raise ValueError(f"model_type must be one of {sorted(valid_model_types)}.")

    dataset_types = {
        "org": False,
        "synthsr": True,
    }

    dfs = []

    for dataset, is_synth in dataset_types.items():
        if dataset not in files_agg:
            raise KeyError(f"Dataset '{dataset}' missing from files_agg.")

        if skin_model not in files_agg[dataset]:
            raise KeyError(f"Skin model '{skin_model}' missing from files_agg['{dataset}'].")

        if model_type not in files_agg[dataset][skin_model]:
            raise KeyError(f"Model type '{model_type}' missing from files_agg['{dataset}']['{skin_model}'].")

        fn = files_agg[dataset][skin_model][model_type]

        print(f"\nLoading {'SynthSR' if is_synth else 'original'} {skin_model}/{model_type}: ({fn})")

        df_stats = load_aparcaseg_table(fn)

        print(f"  Modeled scans: {df_stats['subjid'].nunique()}")
        print(f"  Clinical subjects represented: {df_stats['subjid_base'].nunique()}")

        df_model_dataset = add_roi_stats(df_model=df_model, df_stats=df_stats)
        df_model_dataset["is_synth"] = bool(is_synth)

        dfs.append(df_model_dataset)

    if not dfs:
        raise RuntimeError("No modeling datasets were loaded.")

    df_model_roi = pd.concat(dfs, ignore_index=True)
    df_model_roi["is_synth"] = df_model_roi["is_synth"].astype(bool)

    return df_model_roi


def add_subject_info(
    df: pd.DataFrame,
    df_subject_info: pd.DataFrame,
    age_aggregation: str = "min") -> pd.DataFrame:

    df = df.copy()
    df_subject_info = df_subject_info.copy()

    if "subjid_base" not in df.columns:
        raise KeyError("df must contain 'subjid_base'.")

    if "subjid_base" not in df_subject_info.columns:
        if "subjid" not in df_subject_info.columns:
            raise KeyError("df_subject_info must contain either 'subjid_base' or 'subjid'.")

        df_subject_info = add_base_subjid(df_subject_info, source_col="subjid")

    required_cols = ["subjid_base", "age", "sex"]
    missing_cols = [col for col in required_cols if col not in df_subject_info.columns]

    if missing_cols:
        raise KeyError(f"Subject info missing required columns: {missing_cols}")

    if age_aggregation not in {"min", "max", "mean", "first"}:
        raise ValueError("`age_aggregation` must be 'min', 'max', 'mean', or 'first'.")

    subject_info = df_subject_info[required_cols].copy()

    for column in ("subjid_base", "age", "sex"):
        if subject_info[column].isna().any():
            bad_rows = subject_info.index[subject_info[column].isna()].tolist()
            raise ValueError(f"Missing '{column}' values in subject info at rows: {bad_rows}")

    subject_info["age"] = pd.to_numeric(subject_info["age"], errors="raise")

    sex_conflicts = subject_info.groupby("subjid_base")["sex"].nunique(dropna=False).gt(1)
    conflicting_subjects = sex_conflicts[sex_conflicts].index.tolist()

    if conflicting_subjects:
        raise ValueError(f"Conflicting sex values found for clinical subjects: {conflicting_subjects}")

    age_by_subject = subject_info.groupby("subjid_base")["age"].agg(age_aggregation).rename("age")
    sex_by_subject = subject_info.groupby("subjid_base")["sex"].first().rename("sex")

    subject_info_unique = pd.concat([age_by_subject, sex_by_subject], axis=1).reset_index()

    modeled_subjects = set(df["subjid_base"].dropna())
    info_subjects = set(subject_info_unique["subjid_base"].dropna())

    missing_info = sorted(modeled_subjects - info_subjects)

    if missing_info:
        raise ValueError(f"Missing subject-info records for clinical subjects: {missing_info}")

    return df.merge(subject_info_unique, on="subjid_base", how="left", validate="many_to_one")


def load_subject_info(dataset_name: str) -> pd.DataFrame:
    if dataset_name not in ALLOWED_DATASETS:
        raise ValueError(f"Unknown dataset '{dataset_name}'. Allowed datasets: {ALLOWED_DATASETS}")

    info_path = DATASET_DIR / dataset_name / "subject_info.data"

    if not info_path.is_file():
        raise FileNotFoundError(f"Subject info file does not exist: {info_path}")

    df = pd.read_csv(info_path, sep=",")

    required_cols = ["subjid", "age", "sex"]
    missing_required = [col for col in required_cols if col not in df.columns]

    if missing_required:
        raise ValueError(f"{info_path} is missing required columns: {missing_required}")

    optional_drop_cols = [col for col in ("date_birth", "date_exam") if col in df.columns]

    if optional_drop_cols:
        df = df.drop(columns=optional_drop_cols)

    left_cols = ["subjid", "age", "sex"]
    other_cols = [col for col in df.columns if col not in left_cols]

    return df[left_cols + other_cols]



def load_aparcstats(dataset_name: str, results_dir: str | Path, measure: str = "volume") -> pd.DataFrame:

    if measure not in _APARC_MEASURES:
        raise ValueError(f"measure must be one of {sorted(_APARC_MEASURES)}.")

    stats_dir = Path(results_dir) / dataset_name / "stats"

    if not stats_dir.is_dir():
        raise NotADirectoryError(f"Structural stats directory does not exist: {stats_dir}")

    hemis = []

    for hemi in ("lh", "rh"):
        path = stats_dir / f"{hemi}.aparc.{measure}.stats"

        if not path.is_file():
            raise FileNotFoundError(f"Missing aparc stats file: {path}")

        df = pd.read_csv(path)

        if df.empty:
            raise ValueError(f"Aparc stats file is empty: {path}")

        df = df.rename(columns={df.columns[0]: "subjid"})
        df = add_base_subjid(df, source_col="subjid")

        duplicate_mask = df["subjid"].duplicated(keep=False)

        if duplicate_mask.any():
            duplicates = df.loc[duplicate_mask, "subjid"].tolist()
            raise ValueError(f"Duplicate subjects in {path}: {duplicates}")

        renamed_rois = {}

        for column in df.columns:
            if column in {"subjid", "subjid_base"}:
                continue

            roi = column.removeprefix(f"{hemi}_").removesuffix(f"_{measure}")
            renamed_rois[column] = f"ctx-{hemi}-{roi}"

        df = df.rename(columns=renamed_rois)

        roi_cols = [col for col in df.columns if col not in {"subjid", "subjid_base"}]
        df[roi_cols] = df[roi_cols].apply(pd.to_numeric, errors="raise")

        hemis.append(df)

    lh, rh = hemis

    lh_subjects = set(lh["subjid"])
    rh_subjects = set(rh["subjid"])

    only_lh = sorted(lh_subjects - rh_subjects)
    only_rh = sorted(rh_subjects - lh_subjects)

    if only_lh or only_rh:
        raise ValueError(
            "Left/right aparc subject mismatch.\n"
            f"Only in LH: {only_lh}\n"
            f"Only in RH: {only_rh}"
        )

    return lh.merge(rh, on=["subjid", "subjid_base"], how="inner", validate="one_to_one")



# Main

In [ ]:
DATASET_DIR = _require_env_path("DATASET_DIR")
RESULTS_DIR = _require_env_path("RESULTS_DIR_LARGE")
RESULTS_ROOT = RESULTS_DIR

files_agg = {
    "org": get_stats_table_files("/projects/b32903/Alex/results/STU00225089"),
    "synthsr": get_stats_table_files("/projects/b32903/Alex/results/STU00225089_synthsr"),
}

# CGI scores
df_cog = load_cogscores_batch(
    "/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_agg_2_clean.xlsx"
)

# For now df_model is simply the cognitive-score dataframe.
# No stimulus-data restriction is applied.
df_model = df_cog.copy()

# Load original + SynthSR modeled E-field subjects.
df_model_roi = build_model_roi(
    df_model=df_model,
    files_agg=files_agg,
    skin_model="skin_single",
    model_type="static",
)

# Add age / sex independently of df_stim and df_cog coverage.
df_subject_info = load_subject_info("STU00225089")
df_subject_info = add_base_subjid(df_subject_info, source_col="subjid")
df_model_roi    = add_subject_info(df_model_roi, df_subject_info)


# ============================================================================
# CHECKS
# ============================================================================

print("\nRows by modeling dataset:")
print(df_model_roi["is_synth"].value_counts(dropna=False))

print("\nUnique modeled subjects by modeling dataset:")
print(df_model_roi.groupby("is_synth")["subjid"].nunique())

print("\nUnique clinical subjects by modeling dataset:")
print(df_model_roi.groupby("is_synth")["subjid_base"].nunique())

# print("\nModeled subjects without cognitive-score rows:")
# print(df_model_roi.loc[df_model_roi["subjid_cog"].isna(), ["subjid_base"]].drop_duplicates().values)

# print("\nDataframe shape:")
# print(df_model_roi.shape)
# df_model_roi.to_csv('data_model_roi.csv')

# ============================================================================
# OPTIONAL STIMULUS DATA
#
# Deliberately disabled for now.
# Use later when a specific analysis requires stimulus/session data.
# ============================================================================

# df_stim = load_stimulus(
#     Path("/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/qual_results_june06_2026.xlsx")
# )
#
# df_stim = df_compute_ef_sf(df_stim)
#
# df_model_stim = df_combine_stim_cog(
#     df_stim,
#     df_cog,
#     output_path="data_model.csv",
# )


# ============================================================================
# OPTIONAL STRUCTURAL DATA
# ============================================================================

df_aparc_volume = load_aparcstats(
    dataset_name="STU00225089",
    results_dir=RESULTS_ROOT,
    measure="volume",
)

# df_aparc_thickness = load_aparcstats(
#     dataset_name="STU00225089",
#     results_dir=RESULTS_ROOT,
#     measure="thickness",
# )

# df_aparc_meancurv = load_aparcstats(
#     dataset_name="STU00225089",
#     results_dir=RESULTS_ROOT,
#     measure="meancurv",
# )

In [ ]:
df_aparc_volume

In [ ]:
def load_stimulus_new(fn:str=None, sheet_name:str = None):
    if not fn: 
        raise ValueError('filename (fn) must be defined')
    
    if not isinstance(sheet_name, str):
        return pd.read_excel(fn)
    
    return pd.read_excel(fn, sheet_name=sheet_name)

fn_stim = '/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_agg_2_clean.xlsx'
df_stim = load_stimulus_new(fn_stim, sheet_name='stimulus')
df_stim = df_stim[df_stim.series_num != 0] # 0 = maintenance
df_stim = df_stim[df_stim.frequency_hz != 'age'] # 0 = maintenance
df_stim['ef_sf'] = df_stim["pulse_width_ms"] * df_stim["frequency_hz"] * df_stim["percent_charge"]
df_stim

## CGI Summary 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

class FigCGIViolinStartEnd:
    def __init__(self, df: pd.DataFrame):
        self.df = df

    def run(self):
        tmp = self.df[['subjid', 'cgi_start', 'cgi_end', 'sex']].copy()
        tmp['sex'] = tmp['sex'].replace({'M': 'Male', 'F': 'Female'})
        tmp[['cgi_start', 'cgi_end']] = tmp[['cgi_start', 'cgi_end']].apply(pd.to_numeric, errors='coerce')
        
        tmp = tmp.dropna(subset=['cgi_start', 'cgi_end']).reset_index(drop=True)

        if len(tmp) < 2:
            print('Need at least 2 complete start/end pairs.')
            return

        start = tmp['cgi_start'].to_numpy()
        end = tmp['cgi_end'].to_numpy()

        start_mean, start_sd = start.mean(), start.std(ddof=1)
        end_mean, end_sd = end.mean(), end.std(ddof=1)
        global_mean = np.concatenate([start, end]).mean()

        fig, ax = plt.subplots(figsize=(8, 6))

        parts = ax.violinplot([start, end], positions=[1, 2], showmeans=True, showmedians=True, showextrema=True)

        violin_colors = ['tab:blue', 'orange']
        for body, color in zip(parts['bodies'], violin_colors):
            body.set_facecolor(color)
            body.set_edgecolor('black')
            body.set_alpha(0.5)

        for key in ['cbars', 'cmins', 'cmaxes', 'cmeans', 'cmedians']:
            if key in parts:
                parts[key].set_color('black')
                parts[key].set_linewidth(1.2)

        rng = np.random.default_rng(42)
        jitter = rng.uniform(-0.06, 0.06, size=len(tmp))
        x_start, x_end = 1 + jitter, 2 + jitter

        for i in range(len(tmp)):
            ax.plot([x_start[i], x_end[i]], [start[i], end[i]], color='gray', alpha=0.45, linewidth=1)

        for sex, marker, size in [('Male', 'o', 35), ('Female', '*', 80)]:
            mask = tmp['sex'].eq(sex)
            ax.scatter(x_start[mask], start[mask], s=size, marker=marker, alpha=0.8, color='tab:blue', edgecolors='black', linewidths=0.4, label=f'Start — {sex}')
            ax.scatter(x_end[mask], end[mask], s=size, marker=marker, alpha=0.8, color='orange', edgecolors='black', linewidths=0.4, label=f'End — {sex}')

        ax.set_xticks([1, 2])
        ax.set_xticklabels([f'Start\n{start_mean:.1f} ± {start_sd:.1f}', f'End\n{end_mean:.1f} ± {end_sd:.1f}'])
        ax.set_ylabel('CGI')
        ax.set_ylim([1, 8])
        # ax.set_ybound([0, 8])
        ax.set_yticklabels([i for i in range(1,8)])
        ax.set_title('CGI at Start and End')
        ax.grid(True)
        ax.legend()
        plt.tight_layout()
        plt.show()

        print(f'Start:       n={len(start)}, mean={start_mean:.2f}, SD={start_sd:.2f}, var={start.var(ddof=1):.2f}')
        print(f'End:         n={len(end)}, mean={end_mean:.2f}, SD={end_sd:.2f}, var={end.var(ddof=1):.2f}')
        print(f'Global mean: {global_mean:.2f}')

df = df_cog.copy()
df = df[df['batch_num'] != -1]

df = add_subject_info(df_cog, df_subject_info)

FigCGIViolinStartEnd(df.copy()).run()

del df, response_summary

## Next

In [ ]:
# ============================================================================
# STRUCTURAL DATASET CONFIG
# ============================================================================

RESULTS_ROOT = os.getenv("RESULTS_DIR_LARGE")

if not RESULTS_ROOT:
    raise EnvironmentError("Required environment variable 'RESULTS_DIR_LARGE' is not set.")

STRUCTURAL_DATASETS = {
    "org": {"dataset_name": "STU00225089", "is_synth": False},
    "synthsr": {"dataset_name": "STU00225089_synthsr", "is_synth": True},
}


def load_asegstats(dataset_name: str, results_dir: str | Path = RESULTS_ROOT) -> pd.DataFrame:
    results_dir = Path(results_dir)
    fn = results_dir / dataset_name / "stats" / "aseg.stats"

    if not fn.is_file():
        raise FileNotFoundError(f"aseg stats file does not exist: {fn}")

    df = pd.read_csv(fn)

    if "Measure:volume" in df.columns:
        df = df.rename(columns={"Measure:volume": "subjid"})
    elif "subjid" not in df.columns:
        raise KeyError(f"Could not find subject-ID column in {fn}. Expected 'Measure:volume' or 'subjid'.")

    df = add_base_subjid(df, source_col="subjid")

    duplicate_mask = df["subjid"].duplicated(keep=False)

    if duplicate_mask.any():
        duplicates = df.loc[duplicate_mask, "subjid"].tolist()
        raise ValueError(f"Duplicate modeled subject IDs found in {fn}: {duplicates}")

    df["dataset"] = dataset_name

    return df

def load_all_asegstats(results_dir: str | Path = RESULTS_ROOT) -> pd.DataFrame:
    dfs = []

    for source, config in STRUCTURAL_DATASETS.items():
        df = load_asegstats(config["dataset_name"], results_dir=results_dir)
        df["is_synth"] = config["is_synth"]
        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)

    duplicate_mask = df.duplicated(subset=["subjid", "is_synth"], keep=False)

    if duplicate_mask.any():
        duplicates = df.loc[duplicate_mask, ["subjid", "is_synth"]]
        raise ValueError(f"Duplicate modeled subjects after combining datasets:\n{duplicates.to_string(index=False)}")

    return df

def _lmm_load_aparcaseg_structural(results_dir: str | Path = RESULTS_ROOT):
    dfs_volume = []
    dfs_thickness = []
    dfs_meancurv = []

    for source, config in STRUCTURAL_DATASETS.items():
        dataset_name = config["dataset_name"]
        is_synth = config["is_synth"]

        df_volume = load_aparcstats(dataset_name=dataset_name, results_dir=results_dir, measure="volume")
        df_thickness = load_aparcstats(dataset_name=dataset_name, results_dir=results_dir, measure="thickness")
        df_meancurv = load_aparcstats(dataset_name=dataset_name, results_dir=results_dir, measure="meancurv")

        if "subjid_base" not in df_volume.columns:
            df_volume = add_base_subjid(df_volume, source_col="subjid")

        if "subjid_base" not in df_thickness.columns:
            df_thickness = add_base_subjid(df_thickness, source_col="subjid")

        if "subjid_base" not in df_meancurv.columns:
            df_meancurv = add_base_subjid(df_meancurv, source_col="subjid")

        df_volume["is_synth"] = is_synth
        df_thickness["is_synth"] = is_synth
        df_meancurv["is_synth"] = is_synth

        dfs_volume.append(df_volume)
        dfs_thickness.append(df_thickness)
        dfs_meancurv.append(df_meancurv)

    df_volume = pd.concat(dfs_volume, ignore_index=True)
    df_thickness = pd.concat(dfs_thickness, ignore_index=True)
    df_meancurv = pd.concat(dfs_meancurv, ignore_index=True)

    keys = ["subjid", "is_synth"]

    for name, df in {"volume": df_volume, "thickness": df_thickness, "meancurv": df_meancurv}.items():
        duplicate_mask = df.duplicated(subset=keys, keep=False)

        if duplicate_mask.any():
            duplicates = df.loc[duplicate_mask, keys]
            raise ValueError(f"Duplicate modeled subjects in {name} data:\n{duplicates.to_string(index=False)}")

    return df_volume, df_thickness, df_meancurv

def load_subject_info(dataset_name: str) -> pd.DataFrame:
    if dataset_name != "STU00225089":
        raise ValueError(f"Subject info should be loaded from STU00225089, not '{dataset_name}'.")

    root = os.getenv("DS_NMHC_ROOT")

    if not root:
        raise EnvironmentError("Required environment variable 'DS_NMHC_ROOT' is not set.")

    info_path = Path(root) / "subject_info.data"

    if not info_path.is_file():
        raise FileNotFoundError(f"Subject info file does not exist: {info_path}")

    df = pd.read_csv(info_path, sep=",")

    required_cols = ["subjid", "age", "sex"]
    missing_required = [col for col in required_cols if col not in df.columns]

    if missing_required:
        raise ValueError(f"{info_path} is missing required columns: {missing_required}")

    optional_drop_cols = [col for col in ("date_birth", "date_exam") if col in df.columns]

    if optional_drop_cols:
        df = df.drop(columns=optional_drop_cols)

    df = add_base_subjid(df, source_col="subjid")
    df["age"] = pd.to_numeric(df["age"], errors="raise")

    if df[["subjid", "subjid_base", "age", "sex"]].isna().any().any():
        bad_rows = df.index[df[["subjid", "subjid_base", "age", "sex"]].isna().any(axis=1)].tolist()
        raise ValueError(f"Missing subject-info values at rows: {bad_rows}")

    exact_conflicts = df.groupby("subjid")[["age", "sex"]].nunique(dropna=False).gt(1)

    if exact_conflicts.any().any():
        bad_subjects = exact_conflicts.index[exact_conflicts.any(axis=1)].tolist()
        raise ValueError(f"Conflicting subject-info values for exact subject IDs: {bad_subjects}")

    left_cols = ["subjid", "subjid_base", "age", "sex"]
    other_cols = [col for col in df.columns if col not in left_cols]

    return df[left_cols + other_cols]

def df_combine_aseg2info(df_aseg: pd.DataFrame, df_info: pd.DataFrame) -> pd.DataFrame:
    """
    Attach age/sex to modeled scans.

    Matching priority:
        1. Exact modeled ID: subj-cat-067-002 -> subj-cat-067-002
        2. Base subject fallback: subj-cat-067-002 -> subj-cat-067

    Base-ID fallback is only allowed when age and sex are unambiguous.
    """

    df_aseg = df_aseg.copy()
    df_info = df_info.copy()

    required_aseg = {"subjid", "subjid_base", "is_synth"}
    required_info = {"subjid", "subjid_base", "age", "sex"}

    missing_aseg = required_aseg - set(df_aseg.columns)
    missing_info = required_info - set(df_info.columns)

    if missing_aseg:
        raise KeyError(f"df_aseg missing required columns: {sorted(missing_aseg)}")

    if missing_info:
        raise KeyError(f"df_info missing required columns: {sorted(missing_info)}")

    df_info["age"] = pd.to_numeric(df_info["age"], errors="raise")

    if df_info[["subjid", "subjid_base", "age", "sex"]].isna().any().any():
        bad_rows = df_info.index[df_info[["subjid", "subjid_base", "age", "sex"]].isna().any(axis=1)].tolist()
        raise ValueError(f"Missing subject-info values at rows: {bad_rows}")

    # ------------------------------------------------------------------------
    # Exact modeled-ID subject information
    # ------------------------------------------------------------------------

    exact_conflicts = df_info.groupby("subjid")[["age", "sex"]].nunique(dropna=False).gt(1)

    if exact_conflicts.any().any():
        bad_subjects = exact_conflicts.index[exact_conflicts.any(axis=1)].tolist()
        raise ValueError(f"Conflicting subject-info values for exact modeled IDs: {bad_subjects}")

    df_info_exact = df_info.groupby("subjid", as_index=False).agg(
        subjid_base_info=("subjid_base", "first"),
        age=("age", "first"),
        sex=("sex", "first"),
    )

    result = df_aseg.merge(df_info_exact, on="subjid", how="left", validate="many_to_one")

    # Exact ID must also map to the expected base subject.
    mismatch = result["subjid_base_info"].notna() & result["subjid_base_info"].ne(result["subjid_base"])

    if mismatch.any():
        bad = result.loc[mismatch, ["subjid", "subjid_base", "subjid_base_info"]]
        raise ValueError(f"Exact-ID/base-ID mismatch:\n{bad.to_string(index=False)}")

    result = result.drop(columns=["subjid_base_info"])

    # ------------------------------------------------------------------------
    # Base-ID fallback for modeled scans without exact metadata
    # ------------------------------------------------------------------------

    missing_exact = result["age"].isna() | result["sex"].isna()

    if missing_exact.any():
        fallback_subjects = set(result.loc[missing_exact, "subjid_base"])
        df_info_fallback = df_info[df_info["subjid_base"].isin(fallback_subjects)].copy()

        available_fallback_subjects = set(df_info_fallback["subjid_base"])
        no_fallback = sorted(fallback_subjects - available_fallback_subjects)

        if no_fallback:
            raise ValueError(f"No subject-info data exists for base subjects: {no_fallback}")

        age_conflicts = df_info_fallback.groupby("subjid_base")["age"].nunique(dropna=False).gt(1)
        sex_conflicts = df_info_fallback.groupby("subjid_base")["sex"].nunique(dropna=False).gt(1)

        if age_conflicts.any():
            subjects = age_conflicts[age_conflicts].index.tolist()
            raise ValueError(
                "Cannot use base-subject fallback because multiple ages exist and scan chronology is unknown: "
                f"{subjects}"
            )

        if sex_conflicts.any():
            subjects = sex_conflicts[sex_conflicts].index.tolist()
            raise ValueError(f"Cannot use base-subject fallback because conflicting sex values exist: {subjects}")

        df_info_fallback = df_info_fallback.groupby("subjid_base", as_index=False).agg(
            age_fallback=("age", "first"),
            sex_fallback=("sex", "first"),
        )

        result = result.merge(df_info_fallback, on="subjid_base", how="left", validate="many_to_one")
        result["age"] = result["age"].fillna(result["age_fallback"])
        result["sex"] = result["sex"].fillna(result["sex_fallback"])
        result = result.drop(columns=["age_fallback", "sex_fallback"])

    # ------------------------------------------------------------------------
    # Final validation
    # ------------------------------------------------------------------------

    if result[["age", "sex"]].isna().any().any():
        bad = result.loc[result[["age", "sex"]].isna().any(axis=1), ["subjid", "subjid_base", "is_synth"]]
        raise ValueError(f"No subject-info data could be assigned to modeled subjects:\n{bad.to_string(index=False)}")

    if len(result) != len(df_aseg):
        raise RuntimeError(f"Subject-info merge changed row count: {len(df_aseg)} -> {len(result)}")

    left_cols = ["subjid", "subjid_base", "is_synth", "age", "sex"]
    other_cols = [col for col in result.columns if col not in left_cols]

    return result[left_cols + other_cols].sort_values(["is_synth", "subjid"]).reset_index(drop=True)

df_aseg         = load_all_asegstats()
df_subject_info = load_subject_info("STU00225089")
df_aseg         = df_combine_aseg2info(df_aseg, df_subject_info)

df_volume, df_thickness, df_meancurv = _lmm_load_aparcaseg_structural()

fn = 'data_model_roi.csv'
df_stats_field_aparcseg = load_aparcaseg_table(fn)

cols_exclude_tmp = {"subjid", "subjid_base", "Unknown", "# Unknown"}
cols_aparcseg = [col for col in df_stats_field_aparcseg.columns if col not in cols_exclude_tmp]

required_aseg_etiv = ["subjid", "subjid_base", "is_synth", "EstimatedTotalIntraCranialVol"]
required_aparc_etiv = ["subjid", "subjid_base", "is_synth", "ctx-rh-eTIV", "ctx-lh-eTIV"]

missing_aseg_etiv = [col for col in required_aseg_etiv if col not in df_aseg.columns]
missing_aparc_etiv = [col for col in required_aparc_etiv if col not in df_volume.columns]

if missing_aseg_etiv:
    raise KeyError(f"df_aseg missing ETIV columns: {missing_aseg_etiv}")

if missing_aparc_etiv:
    raise KeyError(f"df_volume missing ETIV columns: {missing_aparc_etiv}")

df_aseg_etiv = df_aseg[required_aseg_etiv].copy()
df_aparc_etiv = df_volume[required_aparc_etiv].copy()

df_aparc_etiv["ctx-eTIV"] = df_aparc_etiv["ctx-rh-eTIV"] + df_aparc_etiv["ctx-lh-eTIV"]

merge_keys = ["subjid", "subjid_base", "is_synth"]

duplicate_aseg = df_aseg_etiv.duplicated(subset=merge_keys, keep=False)

if duplicate_aseg.any():
    duplicates = df_aseg_etiv.loc[duplicate_aseg, merge_keys]
    raise ValueError(f"Duplicate modeled subjects in aseg ETIV data:\n{duplicates.to_string(index=False)}")

duplicate_aparc = df_aparc_etiv.duplicated(subset=merge_keys, keep=False)

if duplicate_aparc.any():
    duplicates = df_aparc_etiv.loc[duplicate_aparc, merge_keys]
    raise ValueError(f"Duplicate modeled subjects in aparc ETIV data:\n{duplicates.to_string(index=False)}")

aseg_keys = set(map(tuple, df_aseg_etiv[merge_keys].to_numpy()))
aparc_keys = set(map(tuple, df_aparc_etiv[merge_keys].to_numpy()))

only_aseg = sorted(aseg_keys - aparc_keys)
only_aparc = sorted(aparc_keys - aseg_keys)

if only_aseg or only_aparc:
    raise ValueError(
        "Modeled-subject mismatch between aseg and aparc ETIV data.\n"
        f"Only in aseg ({len(only_aseg)}): {only_aseg}\n"
        f"Only in aparc ({len(only_aparc)}): {only_aparc}"
    )

df_etiv = df_aseg_etiv.merge(df_aparc_etiv, on=merge_keys, how="inner", validate="one_to_one")


# ============================================================================
# CHECKS
# ============================================================================

print("\nASEG modeled subjects:")
print(df_aseg.groupby("is_synth")["subjid"].nunique())

print("\nAPARC modeled subjects:")
print(df_volume.groupby("is_synth")["subjid"].nunique())

print("\nETIV modeled subjects:")
print(df_etiv.groupby("is_synth")["subjid"].nunique())

print("\nUnique base subjects:")
print(df_etiv.groupby("is_synth")["subjid_base"].nunique())

print("\nETIV dataframe shape:")
print(df_etiv.shape)

# Infer Electrode placement (will be standalone)

In [ ]:
fn_stim = '/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_agg_2_clean.xlsx'
df_stim = load_stimulus_new(fn_stim, sheet_name='stimulus')

df_elec = df_stim[['subjid','electrode_placement']]

df_elec = df_elec.drop_duplicates()
df_elec = df_elec.dropna(axis=0)
df_elec